# Hausse de loyer 2026 - Collection Équinoxe

**Objectif :** estimer la hausse de loyer 2026 des six immeubles de Collection Équinoxe
(Daniel-Johnson, Lévesque, Saint-Elzéar, Le Carlyle, Westpark au Québec; The Met en Ontario),
à partir de l'historique réel des baux, avec une méthode expliquée et validée par backtest.

**Réponse en une ligne :** voir la section 1 pour la définition retenue et la section finale pour l'estimation.

## 0. Configuration

Toutes les constantes du notebook sont déclarées ici, une seule fois, avec leur justification.
Aucun nombre n'apparaît « en dur » plus bas : si un seuil change, il change ici.

Les fichiers CSV sources ne sont jamais modifiés; toute transformation se fait en mémoire dans ce notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# --- Fichiers sources (lecture seule) ---
LISTINGS_PATH = "equinoxe_listings.csv"
LEASES_PATH = "equinoxe_lease_history.csv"
CONCESSIONS_PATH = "equinoxe_concessions.csv"
ASKING_PATH = "equinoxe_asking_history.csv"

# --- Clé d'unité ---
# sPropCode + sUnitCode identifie un appartement de façon unique.
# Jamais sSite + sUnitCode : 248 des 1 061 unités entrent en collision.
UNIT_KEY = ["sPropCode", "sUnitCode"]

# --- Appariement de baux consécutifs d'une même unité ---
MIN_GAP_YEARS = 0.5    # écart plus court : généralement le même bail enregistré deux fois
MAX_GAP_YEARS = 2.5    # écart plus long : comparaison trop ancienne pour être équivalente
DAYS_PER_YEAR = 365.25

# --- Codes de sRenewal ---
RENEWAL = 1            # locataire en place qui renouvelle
TURNOVER = 0           # unité libérée puis relouée (relocation)

# --- Périodes ---
FIRST_ANALYSIS_YEAR = 2019   # avant 2019, trop peu de paires renouvellement pour être fiables
LAST_OBSERVED_YEAR = 2025    # les données s'arrêtent au 2025-12-31
FORECAST_YEAR = LAST_OBSERVED_YEAR + 1
BACKTEST_YEARS = [2023, 2024, 2025]

PCT = 100              # conversion ratio -> pourcentage

### Chargement et conversion des dates

On charge les quatre tables, puis on convertit les colonnes de dates une seule fois.
`lease_start_year` (année de `sLeaseFrom`) est l'année à laquelle on rattache un bail :
c'est la date où le nouveau loyer **entre en vigueur**, donc celle qui compte pour les revenus.

In [ ]:
listings = pd.read_csv(LISTINGS_PATH)
leases = pd.read_csv(LEASES_PATH)
concessions = pd.read_csv(CONCESSIONS_PATH)
asking = pd.read_csv(ASKING_PATH)

for name, df in [("listings", listings), ("leases", leases),
                 ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

LEASE_DATE_COLUMNS = ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]
for col in LEASE_DATE_COLUMNS:
    leases[col] = pd.to_datetime(leases[col], errors="coerce")

leases["lease_start_year"] = leases["sLeaseFrom"].dt.year

## 1. Définir la hausse mesurée

« La hausse de loyer 2026 » n'est pas un seul chiffre. Selon ce qu'on mesure, on obtient des réponses très différentes :

| Définition candidate | Ce qu'elle répond |
|---|---|
| Médiane annuelle de tous les baux | Combien coûte le bail « typique » cette année - **mesure surtout le mix d'immeubles** |
| Même unité, loyer contractuel (`sRent`) | De combien le prix inscrit au bail a monté pour un même appartement |
| Même unité, loyer effectif (`sRentEffective`) | De combien l'argent réellement perçu a monté pour un même appartement |
| Renouvellements seulement (`sRenewal = 1`) | Hausse consentie aux locataires en place - encadrée par le TAL / l'Ontario |
| Relocations seulement (`sRenewal = 0`) | Hausse du prix de marché lorsqu'une unité change de locataire |

Pour choisir, on calcule d'abord toutes ces mesures sur l'historique.

### Croissance à unité constante

La fonction ci-dessous compare chaque bail au **bail précédent de la même unité** (clé `UNIT_KEY`),
ne garde que les écarts comparables (entre `MIN_GAP_YEARS` et `MAX_GAP_YEARS`) et annualise la variation
pour que deux baux séparés de 18 mois soient comparables à deux baux séparés de 12 mois.

In [ ]:
def same_unit_growth(lease_df, price_col="sRent", date_col="sLeaseFrom"):
    """Croissance annualisée (%) du loyer, chaque unité comparée à son propre bail précédent.

    Retourne une ligne par paire (bail précédent -> bail courant) avec la colonne `growth_pct`.
    """
    ordered = lease_df.sort_values(UNIT_KEY + [date_col]).copy()
    by_unit = ordered.groupby(UNIT_KEY)
    ordered["prev_price"] = by_unit[price_col].shift(1)
    ordered["prev_date"] = by_unit[date_col].shift(1)
    pairs = ordered.dropna(subset=["prev_price", "prev_date"])

    pairs = pairs.assign(gap_years=(pairs[date_col] - pairs["prev_date"]).dt.days / DAYS_PER_YEAR)
    is_comparable_gap = pairs["gap_years"].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive="neither")
    has_positive_prices = (pairs["prev_price"] > 0) & (pairs[price_col] > 0)
    pairs = pairs[is_comparable_gap & has_positive_prices].copy()

    price_ratio = pairs[price_col] / pairs["prev_price"]
    pairs["growth_pct"] = (price_ratio ** (1 / pairs["gap_years"]) - 1) * PCT
    return pairs


contract_pairs = same_unit_growth(leases, price_col="sRent")
effective_pairs = same_unit_growth(leases, price_col="sRentEffective")
print(f"{len(contract_pairs):,} paires comparables sur {contract_pairs.groupby(UNIT_KEY).ngroups:,} unités")

### Tableau comparatif des définitions candidates

Médiane par année de début de bail. La médiane plutôt que la moyenne : quelques paires extrêmes
(relocation après une longue occupation, saisie atypique) ne doivent pas déplacer le chiffre.

In [ ]:
analysis_years = range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1)


def median_growth_by_year(pairs, renewal_flag=None):
    subset = pairs if renewal_flag is None else pairs[pairs["sRenewal"] == renewal_flag]
    return subset.groupby("lease_start_year")["growth_pct"].median()


naive_median_rent = leases.groupby("lease_start_year")["sRent"].median()

candidate_definitions = pd.DataFrame({
    "naive_median_all_leases": naive_median_rent.pct_change() * PCT,
    "same_unit_contract": median_growth_by_year(contract_pairs),
    "same_unit_effective": median_growth_by_year(effective_pairs),
    "renewal_contract": median_growth_by_year(contract_pairs, RENEWAL),
    "renewal_effective": median_growth_by_year(effective_pairs, RENEWAL),
    "turnover_contract": median_growth_by_year(contract_pairs, TURNOVER),
    "turnover_effective": median_growth_by_year(effective_pairs, TURNOVER),
}).loc[analysis_years].round(2)

candidate_definitions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for column, style in [("naive_median_all_leases", "o:"),
                      ("same_unit_contract", "o-"),
                      ("same_unit_effective", "s-")]:
    ax.plot(candidate_definitions.index, candidate_definitions[column], style, lw=2, label=column)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("année de début du bail")
ax.set_ylabel("croissance annuelle des loyers (%)")
ax.set_title("Trois définitions de « la hausse », trois histoires différentes")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Définition retenue

> **Hausse 2026 = croissance annualisée du loyer effectif (`sRentEffective`) à unité constante,
> médiane de tous les baux débutant en 2026 (renouvellements et relocations confondus),
> pour les six immeubles.**

**Pourquoi chacun de ces choix :**

1. **À unité constante, pas la médiane annuelle.** Le tableau montre que la médiane naïve s'emballe en 2023
   alors que la croissance à unité constante ralentit cette année-là : la médiane mesure l'arrivée des immeubles
   haut de gamme (Le Carlyle, Westpark, The Met), pas une hausse payée par les locataires. Comparer chaque
   appartement à lui-même neutralise la composition.

2. **Loyer effectif, pas contractuel.** La question d'affaires est de *budgéter les revenus*. Or les revenus sont
   ce qui est réellement perçu après les mois gratuits, le stationnement offert, etc. Les deux mesures évoluent
   ensemble jusqu'en 2023, puis divergent : en 2025, la croissance contractuelle est d'environ 7 % mais
   la croissance effective d'environ 3,6 %. Citer le contractuel à un propriétaire surestimerait la hausse
   de ses revenus de plus de trois points.

3. **Renouvellements et relocations confondus, mais prévus séparément.** Les revenus 2026 proviendront des deux
   types de baux, donc le chiffre final les combine. Mais ils obéissent à des règles différentes
   (plafond TAL / ligne directrice de l'Ontario pour les renouvellements, marché pour les relocations) et
   leur écart change de signe sur la période. On les modélisera donc séparément (section 6), puis on les
   recombinera selon la part attendue de chacun en 2026.

4. **Médiane annualisée, rattachée à l'année de `sLeaseFrom`.** La médiane est robuste aux paires atypiques;
   l'annualisation rend comparables des baux de durées différentes; `sLeaseFrom` est la date où le nouveau
   loyer s'applique.

**Indicateurs secondaires, rapportés à côté du chiffre principal :**

- **Renouvellements, loyer contractuel** : c'est le loyer contractuel que le TAL et la ligne directrice de l'Ontario
  encadrent. C'est donc la bonne mesure pour comparer aux taux réglementaires et pour fixer les loyers de renouvellement.
- **Relocations, loyer contractuel et effectif** : c'est la référence pour fixer les loyers affichés.
- **Par province** : The Met (Ontario) n'est jamais soumis au plafond québécois, ni l'inverse.

**Hypothèses à valider dans les sections suivantes :**

- `sRentEffective` étale correctement les concessions sur la durée du bail (à vérifier contre la table des
  concessions, notamment `PromoPay`, section 5).
- Une paire de baux espacés de 6 à 30 mois compare bien le même produit (aucune rénovation majeure entre les deux).